# Preprocessing e Modelagem — Predição de Alfabetização (execução real)

**Objetivo deste notebook:** orquestrar, de ponta a ponta e contra dados reais
do BigQuery, o pipeline de modelagem definido nas Tasks 1-10: materializar a
tabela Gold enriquecida (aluno + território), dividir os dados em
treino/validação/teste com separação temporal (2023 desenvolvimento, 2024
out-of-time), treinar e selecionar o melhor modelo, avaliar no teste (duas
janelas temporais), diagnosticar variação temporal, interpretar via
SHAP/feature importance, e agregar o risco previsto por município.

Este notebook consome os módulos de `src/` já testados por `pytest` nas
Tasks 1-10 — aqui eles são executados uma única vez, ponta a ponta, contra o
dado de produção (~3,9 milhões de linhas), não contra fixtures. Por isso não
é coberto por `pytest`: sua validação é a própria execução sem erro,
descrita na Task 12 do plano
(`docs/superpowers/specs/2026-09-11-pipeline-modelagem-design.md`).

As células markdown seguem o mesmo formato **Motivo → Resultado → Decisão**
usado no notebook de EDA (`01_eda_gold_e_alunos.ipynb`), sempre que uma etapa
produzir um resultado que muda ou confirma uma decisão de modelagem.


In [1]:
import os
import sys
from pathlib import Path

# O kernel do Jupyter inicia com cwd = diretório do notebook (notebooks/),
# mas os módulos em `src/` e os caminhos usados neste notebook (ex.:
# "reports/...") são relativos à raiz do projeto — subimos um nível e
# adicionamos a raiz ao sys.path antes de qualquer `from src...`.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from google.cloud import bigquery

pd.set_option("display.max_columns", 50)

# `src.visualization.graficos` força o backend "Agg" (matplotlib.use) para
# não exigir display gráfico em testes/CI — como isso reconfigura o backend
# do matplotlib para o processo inteiro, reativamos o backend inline DEPOIS
# de importar os módulos do projeto, para os gráficos aparecerem no notebook.
%matplotlib inline

PROJECT_ID = "fiapfase2"
client = bigquery.Client(project=PROJECT_ID)
print("Projeto BigQuery:", client.project)


Projeto BigQuery: fiapfase2


## 1. Materialização da tabela Gold enriquecida (aluno + território)

In [2]:
from src.preprocessing.gold_materialization import materializar_tabela_enriquecida

# Materialização real — grava (WRITE_TRUNCATE) a tabela
# `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido`. Não é uma
# célula para repetir a cada execução do notebook: é uma materialização de
# camada Gold versionada por código (o código-fonte é o que muda; a tabela é
# regravada quando o código muda), documentada em
# ensinamentos/02-modelagem/01-materializacao-de-camada-gold-versionada.md.
materializar_tabela_enriquecida(client)
print("Tabela materializada:", "fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido")


/Users/pedrosaraiva/FIAP-Fase3/.worktrees/pipeline-modelagem/venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabela materializada: fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** o modelo trabalha na granularidade do aluno (decisão da EDA,
>   notebook 01), mas os microdados de aluno (`basedosdados.br_inep_...alunos`)
>   não têm nenhuma variável territorial/de meta — só a camada Gold da Fase 2
>   (`indicador_por_municipio`) tem `taxa_alfabetizacao`, `gap_meta_resultado` e
>   `meta_alfabetizacao_2024` por município/ano. Sem uma tabela que junte as
>   duas, cada notebook teria que refazer esse enriquecimento (e o join tem uma
>   regra sutil de evitar vazamento — ver abaixo) na mão.
> - **Resultado:** `materializar_tabela_enriquecida` roda a extração+join uma
>   única vez e grava o resultado em
>   `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido` — uma tabela
>   "Gold" nova, na granularidade de aluno, contendo `alfabetizado` (target
>   bruto) e as variáveis territoriais do **ano anterior** ao ano do aluno
>   (`taxa_alfabetizacao_ano_anterior`, `gap_meta_resultado_ano_anterior`,
>   `meta_alfabetizacao_ano_anterior`) — usar o indicador do ano anterior (e
>   não do próprio ano) evita vazamento de dados, porque o indicador do ano
>   corrente é calculado a partir do resultado dos próprios alunos daquele
>   ano (ver `ensinamentos/01-eda/04-vazamento-de-dados-e-dados-ausentes.md`
>   e `ensinamentos/02-modelagem/01-materializacao-de-camada-gold-versionada.md`).
> - **Decisão:** todo o restante do notebook lê apenas essa tabela
>   materializada — nenhum outro notebook/módulo repete esse join.


## 2. Carregar dado, preparar o target e dividir em treino/validação/teste

In [3]:
from src.preprocessing.features import preparar_target
from src.preprocessing.splitting import dividir_treino_validacao_teste

df = client.query(
    "SELECT * FROM `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido`"
).to_dataframe()
df["em_risco"] = preparar_target(df)

partes = dividir_treino_validacao_teste(df)
X_treino, y_treino = partes["treino"], partes["treino"]["em_risco"]
X_validacao, y_validacao = partes["validacao"], partes["validacao"]["em_risco"]
X_teste_mesmo_ano, y_teste_mesmo_ano = partes["teste_mesmo_ano"], partes["teste_mesmo_ano"]["em_risco"]
X_teste_futuro, y_teste_futuro = partes["teste_futuro"], partes["teste_futuro"]["em_risco"]

print("df.shape:", df.shape)
print("em_risco (geral):", df["em_risco"].mean())
print("em_risco por ano:")
print(df.groupby("ano")["em_risco"].mean())
print()
for nome, parte in partes.items():
    print(f"{nome}: {parte.shape[0]} linhas, em_risco={parte['em_risco'].mean():.4f}")


df.shape: (3355846, 11)
em_risco (geral): 0.4086301933998163
em_risco por ano:
ano
2023    0.416239
2024    0.402458
Name: em_risco, dtype: Float64

treino: 1052140 linhas, em_risco=0.4162
validacao: 225459 linhas, em_risco=0.4162
teste_mesmo_ano: 225459 linhas, em_risco=0.4162
teste_futuro: 1852788 linhas, em_risco=0.4025


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** a EDA (notebook 01) estimou a proporção de alunos não
>   alfabetizados em ~41% (e alfabetizados ~59%) a partir dos microdados
>   brutos filtrados por presença — antes de qualquer join com a camada
>   territorial. Como `em_risco = 1 - alfabetizado`, essa é a checagem de
>   sanidade natural para confirmar que a materialização (Etapa 1) e o join
>   não introduziram viés de seleção (ex.: perda de linhas em municípios sem
>   indicador do ano anterior).
> - **Resultado:** ver números impressos acima (`df.shape`, `em_risco` geral
>   e por ano) — preencher/confirmar após a execução real.
> - **Decisão:** se a proporção geral continuar próxima de ~41% e for
>   estável entre 2023/2024, seguimos com o split temporal como definido
>   (70/15/15 dentro de 2023 para treino/validação/teste-mesmo-ano,
>   2024 inteiro como teste out-of-time,
>   `ensinamentos/02-modelagem/02-split-temporal-out-of-time-validation.md`).
>   Um desvio grande sinalizaria problema no join (ex.: municípios sem
>   indicador do ano anterior sendo descartados de forma não aleatória) e
>   pararíamos para investigar antes de treinar qualquer modelo.


## 3. Treinar candidatos e selecionar o vencedor (critério: PR-AUC na validação)

In [4]:
from src.preprocessing.features import build_preprocessor
from src.modeling.candidates import construir_candidatos
from src.modeling.selection import selecionar_melhor_modelo

candidatos = construir_candidatos(build_preprocessor())
nome_vencedor, modelo_vencedor, tabela_selecao = selecionar_melhor_modelo(
    candidatos, X_treino, y_treino, X_validacao, y_validacao,
)
display(tabela_selecao)
print("Modelo vencedor:", nome_vencedor)


,modelo,pr_auc_validacao
0,hist_gradient_boosting,0.598506
1,regressao_logistica,0.596894
2,random_forest,0.593327


Modelo vencedor: hist_gradient_boosting


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** PR-AUC (`average_precision_score`) é o critério de seleção
>   porque a classe de interesse (`em_risco=1`) é minoritária (~41%) e é a
>   classe que importa para a decisão de negócio (priorizar municípios/alunos
>   em risco) — ROC-AUC é otimista demais em cenários com esse desbalanceamento
>   moderado, porque pondera bem também o desempenho na classe majoritária.
> - **Resultado:** ver `tabela_selecao` acima — modelo vencedor e PR-AUC de
>   validação de cada candidato (regressão logística, random forest,
>   hist gradient boosting) — preencher após a execução real.
> - **Decisão:** **esta é a única célula do notebook em que números de
>   validação aparecem no relatório final** — a partir daqui, toda métrica
>   reportada vem exclusivamente dos conjuntos de teste (mesmo-ano e
>   out-of-time), nunca mais da validação, para não recontaminar a escolha
>   do modelo com informação de teste
>   (`ensinamentos/02-modelagem/04-disciplina-validacao-vs-teste-no-loop-de-iteracao.md`).


## 4. Avaliação final no teste — 2023 (mesmo ano) e 2024 (out-of-time) — uma única vez

In [5]:
from src.evaluation.metricas import calcular_metricas_teste, tabela_limiares

metricas_teste_mesmo_ano = calcular_metricas_teste(modelo_vencedor, X_teste_mesmo_ano, y_teste_mesmo_ano)
metricas_teste_futuro = calcular_metricas_teste(modelo_vencedor, X_teste_futuro, y_teste_futuro)

print("Teste-2023 (mesmo ano):", metricas_teste_mesmo_ano)
print("2024 (out-of-time):", metricas_teste_futuro)

limiares_teste_futuro = tabela_limiares(modelo_vencedor, X_teste_futuro, y_teste_futuro)
display(limiares_teste_futuro)


Teste-2023 (mesmo ano): {'roc_auc': 0.6885271401484355, 'pr_auc': 0.5968419815001267}
2024 (out-of-time): {'roc_auc': 0.6379186574876897, 'pr_auc': 0.5269328353706211}


,cenario,limiar,precisao,recall
0,padrao (limiar 0.5),0.500000,0.480685,0.667373
1,otimizado para F1,0.358403,0.438322,0.911721
2,recall-prioritario (recall >= 80%),0.428667,0.459625,0.801681


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** o teste-mesmo-ano (2023, held-out do mesmo ano de treino) mede
>   a capacidade de generalização "normal" do modelo; o teste-2024
>   (out-of-time, um ano inteiro nunca visto em treino/validação) mede a
>   capacidade de generalizar para o futuro — o cenário real de uso do
>   modelo (prever risco no ano seguinte ao treinado).
> - **Resultado:** ver métricas (ROC-AUC/PR-AUC) e a tabela de limiares
>   (padrão 0.5, otimizado para F1, priorizando recall ≥ 80%) impressas acima
>   — preencher após a execução real.
> - **Decisão:** interpretar o gap entre as duas janelas —
>   preencher após ver os números reais (ver Etapa 5, que classifica a causa
>   do gap, e
>   `ensinamentos/02-modelagem/02-split-temporal-out-of-time-validation.md`).


## 5. Diagnóstico de variação temporal (2023 vs. 2024)

In [6]:
from src.evaluation.variacao_temporal import comparar_distribuicoes_categoricas, comparar_distribuicoes_numericas
from src.preprocessing.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES

df_2023 = df[df["ano"] == 2023]
df_2024 = df[df["ano"] == 2024]

variacao_numerica = comparar_distribuicoes_numericas(df_2023, df_2024, NUMERIC_FEATURES)
variacao_categorica = comparar_distribuicoes_categoricas(df_2023, df_2024, CATEGORICAL_FEATURES)

display(variacao_numerica)
display(variacao_categorica)


,coluna,estatistica_ks,p_valor,distribuicao_mudou
0,taxa_alfabetizacao_ano_anterior,0.067675,0.0,True
1,gap_meta_resultado_ano_anterior,0.072888,0.0,True
2,meta_alfabetizacao_ano_anterior,0.072956,0.0,True


,coluna,categoria,proporcao_2023,proporcao_2024,diferenca_absoluta
0,rede,Estadual,0.087506,0.130168,0.042662
1,rede,Municipal,0.912494,0.869819,0.042675
2,rede,Privada,0.000000,0.000013,0.000013
3,regiao,Centro-Oeste,0.105139,0.084446,0.020693
4,regiao,Nordeste,0.334998,0.256638,0.078361
5,regiao,Norte,0.120280,0.097985,0.022295
6,regiao,Sudeste,0.240994,0.407933,0.166939
7,regiao,Sul,0.198589,0.152999,0.045591
8,sigla_uf,AL,0.021839,0.017393,0.004446
9,sigla_uf,AM,0.031977,0.025684,0.006292


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** um gap entre teste-2023 e teste-2024 pode ter duas causas bem
>   diferentes — *covariate shift* (a distribuição das features de entrada
>   mudou, mas a relação feature→target continua a mesma) ou *concept shift*
>   (a própria relação feature→target mudou) — e a ação corretiva é diferente
>   em cada caso (ver
>   `ensinamentos/02-modelagem/03-covariate-shift-vs-concept-shift.md`).
>   O teste de Kolmogorov-Smirnov (`estatistica_ks`/`p_valor`) nas features
>   numéricas e a comparação de proporções nas categóricas são a forma de
>   testar a primeira hipótese sem precisar de rótulo (`em_risco`) de 2024.
> - **Resultado:** ver tabelas `variacao_numerica` e `variacao_categorica`
>   acima — preencher após a execução real com quais colunas mudaram de
>   distribuição (`distribuicao_mudou=True`) e a magnitude
>   (`diferenca_absoluta`) das categóricas.
> - **Decisão:** classificar o gap da Etapa 4 — preencher após ver os
>   números reais: se as features não mudaram de distribuição, o gap (se
>   houver) é evidência de concept shift; se mudaram bastante, é (ao menos
>   em parte) covariate shift.


## 6. Interpretabilidade — importância de features e SHAP

In [7]:
from src.modeling.interpretabilidade import calcular_shap_values
from src.visualization.graficos import plot_importancia_features, plot_curva_precisao_recall

preprocessador_ajustado = modelo_vencedor.named_steps["preprocessamento"]
classificador_vencedor = modelo_vencedor.named_steps["classificador"]
nomes_features = list(preprocessador_ajustado.get_feature_names_out())

# SHAP: amostra (custo computacional), não o dataset completo — ver nota
# sobre amostragem para SHAP em
# ensinamentos/02-modelagem/04-disciplina-validacao-vs-teste-no-loop-de-iteracao.md.
# Calculado ANTES da importância porque, para HistGradientBoostingClassifier,
# o SHAP é a única fonte de importância global disponível (ver célula abaixo).
amostra_shap = X_teste_futuro.sample(n=min(3000, len(X_teste_futuro)), random_state=42)
valores_shap = calcular_shap_values(modelo_vencedor, amostra_shap)


**Checagem de sanidade da correção (commit `f815d56`):** antes da tentativa
anterior, as 3 features numéricas territoriais ficavam 100% nulas no
cohort de treino (2023), porque `REDE_REFERENCIA_TERRITORIO` era
"Pública (Estadual e Municipal)" (sem `gap_meta_resultado`/
`meta_alfabetizacao_2024` na Gold real) e o join usava sempre `ano-1`
(2022, inexistente na Gold). A correção trocou a rede de referência para
"Municipal" e adicionou fallback para o mesmo ano quando `ano-1` não
existir. A célula abaixo verifica isso diretamente nos dados de treino
antes de prosseguir para o SHAP — se ainda houver colunas 100% nulas, o
`SimpleImputer` as descarta silenciosamente do pipeline (comportamento
observado na tentativa anterior).


In [8]:
from src.preprocessing.features import NUMERIC_FEATURES

percentual_nulos_treino = X_treino[NUMERIC_FEATURES].isna().mean()
print("Percentual de nulos nas features numéricas (X_treino, cohort 2023):")
print(percentual_nulos_treino)


Percentual de nulos nas features numéricas (X_treino, cohort 2023):
taxa_alfabetizacao_ano_anterior    0.001139
gap_meta_resultado_ano_anterior    0.041142
meta_alfabetizacao_ano_anterior    0.041142
dtype: float64


In [9]:
import numpy as np
import pandas as pd

# HistGradientBoostingClassifier não expõe `feature_importances_` nem
# `coef_` (só RandomForest e LogisticRegression, respectivamente, expõem —
# lacuna conhecida da API do scikit-learn para esse modelo). Para os três
# candidatos funcionarem com o mesmo código, usamos a importância nativa
# quando disponível e, quando não, a média do |valor SHAP| por feature
# (já calculado na célula anterior) como substituto — mesma unidade
# conceitual (contribuição média para a predição).
if hasattr(classificador_vencedor, "feature_importances_"):
    importancias = classificador_vencedor.feature_importances_
elif hasattr(classificador_vencedor, "coef_"):
    importancias = abs(classificador_vencedor.coef_[0])
else:
    importancias = np.abs(valores_shap.values).mean(axis=0)

tabela_importancia = (
    pd.DataFrame({"feature": nomes_features, "importancia": importancias})
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
print("Top 10 features mais importantes:")
print(tabela_importancia.head(10).to_string(index=False))

fig_importancia = plot_importancia_features(nomes_features, importancias)
fig_importancia.savefig("reports/importancia_features.png", dpi=100, bbox_inches="tight")

fig_pr = plot_curva_precisao_recall(modelo_vencedor, X_teste_futuro, y_teste_futuro)
fig_pr.savefig("reports/curva_precisao_recall.png", dpi=100, bbox_inches="tight")

import shap
import matplotlib.pyplot as plt

# Nota: `valores_shap` é um shap.Explanation calculado sobre os dados JÁ
# transformados pelo preprocessador (one-hot expande as 3 categóricas em
# várias colunas) — por isso NÃO passamos `amostra_shap` (dataframe bruto,
# menos colunas) como segundo argumento: o shape não bateria com
# `valores_shap.values`. Passar só o Explanation deixa o shap usar seus
# próprios `.data`/`.feature_names` (já no espaço transformado).
shap.summary_plot(valores_shap, show=False)
plt.savefig("reports/shap_summary.png", dpi=100, bbox_inches="tight")
plt.close()

print("Gráficos salvos em reports/importancia_features.png, reports/curva_precisao_recall.png, reports/shap_summary.png")


Top 10 features mais importantes:
                                   feature  importancia
numericas__taxa_alfabetizacao_ano_anterior     0.549053
                categoricas__rede_Estadual     0.031495
numericas__gap_meta_resultado_ano_anterior     0.030177
               categoricas__rede_Municipal     0.016545
numericas__meta_alfabetizacao_ano_anterior     0.012919
              categoricas__regiao_Nordeste     0.009906
               categoricas__regiao_Sudeste     0.007274
          categoricas__regiao_Centro-Oeste     0.003429
                  categoricas__sigla_uf_AM     0.003341
                   categoricas__regiao_Sul     0.003211


Gráficos salvos em reports/importancia_features.png, reports/curva_precisao_recall.png, reports/shap_summary.png


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** feature importance (do próprio classificador) dá uma visão
>   rápida e global de quais variáveis o modelo mais usa; SHAP complementa
>   mostrando também a *direção* do efeito de cada feature (valores altos de
>   uma variável aumentam ou diminuem o risco previsto) — informação que a
>   importância bruta não dá, e que é o que o público não técnico
>   (stakeholders do setor público) vai perguntar: "por que este município é
>   classificado como risco alto?".
> - **Resultado:** ver gráficos acima — preencher com as features de maior
>   importância/maior impacto SHAP após a execução real.
> - **Decisão:** as features de maior impacto (preencher) são as que o
>   README (Task 13) deve destacar na resposta à pergunta de negócio "quais
>   fatores mais afetam a alfabetização".


## 7. Agregação de risco por município e artefatos finais

In [10]:
import joblib
from src.evaluation.risco_por_municipio import agregar_risco_por_municipio

probabilidades_futuro = modelo_vencedor.predict_proba(X_teste_futuro)[:, 1]
risco_por_municipio = agregar_risco_por_municipio(X_teste_futuro["id_municipio"], probabilidades_futuro)
display(risco_por_municipio.head(20))

risco_por_municipio.to_csv("reports/risco_por_municipio_2024.csv", index=False)
joblib.dump(modelo_vencedor, "reports/modelo_vencedor.joblib")

print("Artefatos salvos em reports/risco_por_municipio_2024.csv e reports/modelo_vencedor.joblib")


,id_municipio,risco_medio,n_alunos
0,1718501,0.932496,48
1,1716307,0.930388,58
2,1718006,0.930388,47
3,2919900,0.925439,40
4,2406908,0.921087,21
5,2205581,0.920036,55
6,1717800,0.915161,42
7,1715705,0.915161,81
8,2505238,0.911822,65
9,2515005,0.911822,23


Artefatos salvos em reports/risco_por_municipio_2024.csv e reports/modelo_vencedor.joblib


> ## Síntese final (alimenta o README da Task 13)
>
> - **Modelo escolhido:** preencher após a execução real (nome + PR-AUC de
>   validação da Etapa 3).
> - **Métricas de teste:** preencher (ROC-AUC/PR-AUC em teste-2023 vs.
>   2024, Etapa 4) e a classificação do gap (covariate shift / concept
>   shift / sem variação relevante, Etapa 5).
> - **Top-5 municípios de maior risco previsto (2024):** preencher a partir
>   de `risco_por_municipio.head(5)` acima.
> - **Top-5 features mais influentes:** preencher a partir da Etapa 6
>   (importância do classificador + SHAP).
> - **Artefatos gerados:** `reports/risco_por_municipio_2024.csv` (risco
>   médio previsto por município em 2024, ordenado do maior para o menor) e
>   `reports/modelo_vencedor.joblib` (pipeline completo — pré-processamento +
>   classificador — pronto para novas predições).
